# Hierarchical MARL Chip Placement - GPU Training on Google Colab

This notebook trains the hierarchical MARL chip placement models with GPU acceleration.

## 1. Setup & Dependencies

In [ ]:
# Check GPU availability
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

GPU Available: True
GPU Device: Tesla T4
GPU Memory: 15.64 GB


In [ ]:
# Clone the repository
import os
from pathlib import Path

REPO_URL = "https://github.com/Pequanta/hierarchical-marl-chip-placement.git"  # Replace with actual repo URL
WORK_DIR = "/content/hierarchical-marl-chip-placement"

# Clone if not already present
if not Path(WORK_DIR).exists():
    !git clone {REPO_URL} {WORK_DIR}
    print(f"Repository cloned to {WORK_DIR}")
else:
    print(f"Repository already exists at {WORK_DIR}")

os.chdir(WORK_DIR)
print(f"Working directory: {os.getcwd()}")

Cloning into '/content/hierarchical-marl-chip-placement'...
remote: Enumerating objects: 227, done.
remote: Counting objects: 100% (109/109), done.
remote: Compressing objects: 100% (83/83), done.
remote: Total 227 (delta 28), reused 98 (delta 24), pack-reused 118 (from 1)
Receiving objects: 100% (227/227), 26.33 MiB | 18.87 MiB/s, done.
Resolving deltas: 100% (56/56), done.
Repository cloned to /content/hierarchical-marl-chip-placement
Working directory: /content/hierarchical-marl-chip-placement


In [ ]:
!pip install uv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.7/24.7 MB 74.6 MB/s eta 0:00:00


In [ ]:
# Install dependencies
!uv pip install -q -e .
!uv pip install -q -r requirements.txt
print("Dependencies installed successfully!")

Dependencies installed successfully!


## 2. Configuration

In [ ]:
import yaml
from pathlib import Path

# Load and display config files
config_dir = Path("configs")

configs = {}
for config_file in ["env.yaml", "gnn.yaml", "rl.yaml", "training.yaml"]:
    config_path = config_dir / config_file
    if config_path.exists():
        with open(config_path, 'r') as f:
            configs[config_file.replace('.yaml', '')] = yaml.safe_load(f)
        print(f"\nLoaded {config_file}:")
        print(yaml.dump(configs[config_file.replace('.yaml', '')], default_flow_style=False))
    else:
        print(f"Warning: {config_file} not found")


Loaded env.yaml:
action_space:
  encoded_action: macro_index * num_directions + direction_index
  manager_action: macro_index
  movement_step: 0.05
  num_directions: 4
  type: hierarchical_discrete
  worker_action: movement_direction
canvas:
  clamp_positions: true
  coordinate_system: normalized_unit_square
  default_size:
  - 400.0
  - 400.0
class_path: src.models.macro_placement_env.MacroPlacementEnv
dataset:
  connection_type: real-connection
  design: MemPool_tile
  graph_path: src/data/preprocessed/real-connection/MemPool_tile/Nangate45/MemPool_tile_Nangate45_graph.pt
  metadata_path: src/data/preprocessed/real-connection/MemPool_tile/Nangate45/metadata-MemPool_tile-Nangate45.json
  technology: Nangate45
episode:
  initial_position_distribution: uniform
  max_steps: 200
  randomize_initial_positions: true
  seed: 42
name: macro_placement_env
observation:
  dtype: float32
  fields:
  - normalized_x
  - normalized_y
  - node_features
  high: 1.0
  low: 0.0
  normalization:
    nod

## 3. Training Setup

In [ ]:
# Training parameters (customize as needed)
TRAINING_CONFIG = {
    "script": "scripts/train.py",  # Options: "scripts/train.py" or "scripts/train_gnn.py"
    "env_config": "configs/env.yaml",
    "gnn_config": "configs/gnn.yaml",
    "rl_config": "configs/rl.yaml",
    "training_config": "configs/training.yaml",
    "output_dir": "results/",
    "log_dir": "results/logs/",
    "checkpoint_dir": "checkpoints/",
}

print("Training Configuration:")
for key, value in TRAINING_CONFIG.items():
    print(f"  {key}: {value}")

Training Configuration:
  script: scripts/train.py
  env_config: configs/env.yaml
  gnn_config: configs/gnn.yaml
  rl_config: configs/rl.yaml
  training_config: configs/training.yaml
  output_dir: results/
  log_dir: results/logs/
  checkpoint_dir: checkpoints/


In [ ]:
# Create output directories
from pathlib import Path

for dir_path in [TRAINING_CONFIG["output_dir"],
                  TRAINING_CONFIG["log_dir"],
                  TRAINING_CONFIG["checkpoint_dir"]]:
    Path(dir_path).mkdir(parents=True, exist_ok=True)
    print(f"Created/verified directory: {dir_path}")

Created/verified directory: results/
Created/verified directory: results/logs/
Created/verified directory: checkpoints/


## 4. Run Training

In [ ]:
import subprocess
import sys

TRAINING_CONFIG = {
    "script": "scripts/train.py",
    "rl_config": "configs/rl.yaml",
    "env_config": "configs/env.yaml",
    "training_config": "configs/training.yaml",

    # Optional overrides
    "graph_path": None,
    "algorithm": "ppo",
    "timesteps": 100000,
    "checkpoint_dir": "checkpoints",
    "device": "cuda",
    "seed": 42,
    "use_gnn_encoder": True,

    # Optional outputs
    "save_final_graph": "outputs/final_graph.pt",
    "history_json": "outputs/history.json",
}

# Build command
cmd = [
    sys.executable,
    TRAINING_CONFIG["script"],

    "--rl-config",
    TRAINING_CONFIG["rl_config"],

    "--env-config",
    TRAINING_CONFIG["env_config"],

    "--training-config",
    TRAINING_CONFIG["training_config"],
]

# Optional arguments
if TRAINING_CONFIG.get("graph_path"):
    cmd.extend(["--graph-path", TRAINING_CONFIG["graph_path"]])

if TRAINING_CONFIG.get("algorithm"):
    cmd.extend(["--algorithm", TRAINING_CONFIG["algorithm"]])

if TRAINING_CONFIG.get("timesteps") is not None:
    cmd.extend(["--timesteps", str(TRAINING_CONFIG["timesteps"])])

if TRAINING_CONFIG.get("checkpoint_dir"):
    cmd.extend(["--checkpoint-dir", TRAINING_CONFIG["checkpoint_dir"]])

if TRAINING_CONFIG.get("device"):
    cmd.extend(["--device", TRAINING_CONFIG["device"]])

if TRAINING_CONFIG.get("seed") is not None:
    cmd.extend(["--seed", str(TRAINING_CONFIG["seed"])])

# BooleanOptionalAction handling
if TRAINING_CONFIG.get("use_gnn_encoder") is True:
    cmd.append("--use-gnn-encoder")
elif TRAINING_CONFIG.get("use_gnn_encoder") is False:
    cmd.append("--no-use-gnn-encoder")

if TRAINING_CONFIG.get("save_final_graph"):
    cmd.extend([
        "--save-final-graph",
        TRAINING_CONFIG["save_final_graph"]
    ])

if TRAINING_CONFIG.get("history_json"):
    cmd.extend([
        "--history-json",
        TRAINING_CONFIG["history_json"]
    ])

print("Running training command:")
print(" ".join(cmd))
print("\n" + "=" * 80 + "\n")

# Run training
result = subprocess.run(
    cmd,
    capture_output=True,
    text=True
)

print(result.stdout)

if result.stderr:
    print("\nSTDERR:")
    print(result.stderr)

print("\nReturn code:", result.returncode)

if result.returncode == 0:
    print("\n" + "=" * 80)
    print("Training completed successfully!")
else:
    print("\n" + "=" * 80)
    print(f"Training failed with return code: {result.returncode}")

Running training command:
/usr/bin/python3 scripts/train.py --rl-config configs/rl.yaml --env-config configs/env.yaml --training-config configs/training.yaml --algorithm ppo --timesteps 100000 --checkpoint-dir checkpoints --device cuda --seed 42 --use-gnn-encoder --save-final-graph outputs/final_graph.pt --history-json outputs/history.json


Resolved training config: {'graph_path': '/content/hierarchical-marl-chip-placement/src/data/preprocessed/real-connection/MemPool_tile/Nangate45/MemPool_tile_Nangate45_graph.pt', 'algorithm': 'ppo', 'total_timesteps': 100000, 'rollout_steps': 2048, 'eval_frequency': 10000, 'eval_episodes': 3, 'seed': 42, 'device': 'gpu', 'checkpoint_dir': '/content/hierarchical-marl-chip-placement/checkpoints', 'save_best_graph': True, 'num_directions': 4, 'env_config': {'max_steps': 200, 'movement_step': 0.05, 'hpwl_scale': 0.001, 'randomize_initial_positions': True, 'include_step_fraction': False, 'overlap_weight': 0.0, 'density_weight': 0.0}, 'eval_env_config': 

## 5. Monitor Results

In [ ]:
from pathlib import Path
import json

# List checkpoint files
checkpoint_dir = Path(TRAINING_CONFIG["checkpoint_dir"])
if checkpoint_dir.exists():
    checkpoints = list(checkpoint_dir.glob("*.pt"))
    print(f"Found {len(checkpoints)} checkpoint files:")
    for ckpt in sorted(checkpoints):
        print(f"  - {ckpt.name}")
else:
    print("Checkpoint directory not found")

# List log files
log_dir = Path(TRAINING_CONFIG["log_dir"])
if log_dir.exists():
    logs = list(log_dir.glob("*.log")) + list(log_dir.glob("*.json"))
    print(f"\nFound {len(logs)} log files:")
    for log in sorted(logs):
        print(f"  - {log.name}")
else:
    print("\nLog directory not found")

Found 3 checkpoint files:
  - best_placement_step_100000.pt
  - best_ppo.pt
  - final_ppo.pt


KeyError: 'log_dir'

In [ ]:
from google.colab import files
from pathlib import Path
import shutil

# Download results
results_dir = Path("/content/hierarchical-marl-chip-placement/outputs/")
if results_dir.exists():
    # Create archive of results
    archive_path = shutil.make_archive("training_results", "zip", results_dir)
    print(f"Downloading results from {results_dir}...")
    files.download(archive_path)
    print("Download complete!")
else:
    print(f"Results directory {results_dir} not found")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download complete!


## Troubleshooting

If you encounter issues:

1. **GPU not available**: Go to Runtime → Change runtime type and select GPU
2. **Import errors**: Check that the repository URL is correct and the project structure is as expected
3. **Missing config files**: Verify that config files exist in the `configs/` directory
4. **Training script arguments**: Modify the `cmd` list in cell 4 to match your script's expected arguments

### Alternative: Manual training command

You can also run training manually with custom arguments:

In [ ]:
# Uncomment and modify as needed for your specific training requirements
# !cd {WORK_DIR} && python scripts/train.py --help